In [ ]:
!pip install langchain langchain-community langchain-google-genai pageindex rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [ ]:
pi_client = PageIndexClient(api_key=userdata.get('PAGEINDEX'))

In [ ]:
#Corpus Preparation

import requests
from bs4 import BeautifulSoup
import json
import re
import nltk

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)


# A list of thematically related UN Women / UNICEF / human-rights articles.

URLS = [
    "https://www.unwomen.org/en/articles/faqs/faqs-afghanistan",
    "https://www.unicef.org/afghanistan/education",
    "https://www.ohchr.org/en/countries/afghanistan",
    "https://www.hrw.org/world-report/2024/country-chapters/afghanistan",
    "https://www.amnesty.org/en/location/asia-and-the-pacific/south-asia/afghanistan/report-afghanistan/",
]

def scrape_article(url: str) -> dict:
    """
    Fetch a web page and extract the plain-text body.
    Returns a dict with 'url', 'title', 'raw_text'.
    Falls back gracefully if the page is unavailable.
    """
    try:
        resp = requests.get(url, timeout=15,
                            headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
    except Exception as e:
        print(f"  ⚠ Could not fetch {url}: {e}")
        return None

    soup = BeautifulSoup(resp.text, "html.parser")


    h1 = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else (soup.title.string if soup.title else url)

    # Collect all paragraph text
    paras = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
    raw_text = "\n".join(p for p in paras if len(p) > 40)

    return {"url": url, "title": title, "raw_text": raw_text}


def clean_text(text: str) -> str:
    """
    Standard NLP preprocessing:
      - collapse whitespace / newlines
      - remove non-ASCII garbage characters
      - strip leading/trailing whitespace
    We keep capitalisation and punctuation at this stage
    because BM25 tokenisation handles lowercasing internally.
    """
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)               # collapse whitespace
    return text.strip()


# Scrape all articles and store them as cleaned dicts
corpus_raw = []
for url in URLS:
    print(f"Scraping: {url}")
    art = scrape_article(url)
    if art and len(art["raw_text"]) > 200:
        art["text"] = clean_text(art["raw_text"])
        corpus_raw.append(art)
        print(f"  ✓ '{art['title'][:60]}' — {len(art['text'])} chars")

print(f"\nCorpus size: {len(corpus_raw)} articles")

In [ ]:
from nltk.tokenize import sent_tokenize


# Chunking each article into fixed-size sentence windows.
# This is necessary because:
#   - BM25 provides individual chunks, not complete articles
#   - smaller units give more precise retrieval
# We use a window of 5 sentences with no overlap for simplicity.

CHUNK_SIZE = 5   # no. of sentences per chunk

all_chunks = []

for art in corpus_raw:
    sents = sent_tokenize(art["text"])
    for i in range(0, len(sents), CHUNK_SIZE):
        chunk_text = " ".join(sents[i : i + CHUNK_SIZE])
        if len(chunk_text.strip()) < 30:
            continue
        all_chunks.append({
            "text": chunk_text,
            "source_title": art["title"],
            "source_url": art["url"],
            "chunk_id": len(all_chunks),
        })

print(f"Total chunks: {len(all_chunks)}")
print("\nSample chunk:")
print(all_chunks[0]["text"][:300])

In [ ]:
# Saving the fully cleaned corpus into a single text file - to be uploaded to PageIndex

corpus_path = "corpus.txt"

with open(corpus_path, "w", encoding="utf-8") as f:
    for art in corpus_raw:
        f.write(f"=== {art['title']} ===\n")
        f.write(art["text"])
        f.write("\n\n")

print(f"Saved corpus to {corpus_path}")

with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2)

print(f"Saved {len(all_chunks)} chunks to chunks.json")

Heirarchical PageIndex

In [ ]:
import time
from pageindex import PageIndexClient

# Initialise the PageIndex client with our secret key
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

# Upload the corpus file to PageIndex.
# PageIndex will parse the document and build an internal hierarchical tree:
#   Document → Titles → Sections → Chunks
print("Uploading corpus to PageIndex...")
with open(corpus_path, "rb") as f:
    upload_response = pi_client.upload(f, filename="corpus.txt")

doc_id = upload_response.id
print(f"Upload complete. Document ID: {doc_id}")

# Indexing happens asynchronously on PageIndex servers.
# We poll the status every 10 s until the document is ready.
print("Waiting for indexing to complete...")
while True:
    status = pi_client.get_status(doc_id)
    if status.status == "ready":
        print("✓ Indexing complete!")
        break
    print("  Still indexing...")
    time.sleep(10)

In [ ]:
tree = pi_client.get_tree(doc_id)

# *** Initialise the list before traversal to avoid NameError ***
pageindex_docs = []

def traverse(node, parent_title="", parent_section=""):
    """
    Recursively walk the PageIndex tree.
    When we reach a leaf node (no children), we treat its text as a chunk
    and record it alongside breadcrumb metadata from its ancestors.

    Parameters
    ----------
    node           : current tree node (PageIndex SDK object)
    parent_title   : title of the nearest ancestor Title node
    parent_section : section name of the nearest ancestor Section node
    """
    # Determine what context label to pass down
    current_title   = node.title   if hasattr(node, 'title')   and node.title   else parent_title
    current_section = node.section if hasattr(node, 'section') and node.section else parent_section

    # If this node has text and no children → it is a leaf chunk
    children = getattr(node, 'children', []) or []
    node_text = getattr(node, 'text', '') or ''

    if node_text.strip() and not children:
        pageindex_docs.append({
            "text": node_text.strip(),
            "title": current_title,
            "section": current_section,
        })

    # Recurse into children
    for child in children:
        traverse(child, current_title, current_section)


traverse(tree)

print(f"PageIndex nodes extracted: {len(pageindex_docs)}")
print("\nSample node:")
print(pageindex_docs[0] if pageindex_docs else "(none — check tree structure)")

BM25 Retrieval Implementation

In [ ]:
from rank_bm25 import BM25Okapi
import string

# We use the PageIndex-extracted chunks as our document store

if len(pageindex_docs) > 0:
    bm25_corpus = pageindex_docs
    print("Using PageIndex nodes for BM25")
else:

    bm25_corpus = [{"text": c["text"], "title": c["source_title"], "section": ""}
                   for c in all_chunks]
    print("Fallback: using pre-split chunks for BM25")


def tokenize(text: str) -> list:
    """
    Simple whitespace tokeniser with lowercasing and punctuation removal.
    BM25 works on token frequencies, so consistent tokenisation matters —
    the same function must be used at index time AND query time.
    """
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.split()


# Tokenise every document in the corpus
tokenized_corpus = [tokenize(doc["text"]) for doc in bm25_corpus]

# Build the BM25Okapi index.
# BM25Okapi is the most common variant of BM25; it adds a small saturation
# constant (epsilon) that avoids zero IDF for very common terms.
bm25_index = BM25Okapi(tokenized_corpus)

print(f"BM25 index built over {len(tokenized_corpus)} documents")

In [ ]:
from langchain.schema import BaseRetriever, Document
from pydantic import Field
from typing import List

class BM25Retriever(BaseRetriever):
    """
    A LangChain-compatible retriever backed by a BM25 index.

    LangChain's RAG chain expects any retriever to inherit from BaseRetriever
    and implement the `_get_relevant_documents` method.
    That method takes a plain-text query and returns a list of
    LangChain `Document` objects (each with .page_content and .metadata).

    Pydantic requires us to declare mutable fields (like the BM25 index
    and the document list) using `Field(default=None)` with type annotations
    so that LangChain's BaseRetriever validation doesn't reject them.
    """

    # Declare the BM25 index and doc store as typed Pydantic fields.
    bm25: object = Field(default=None)
    docs: List[dict] = Field(default_factory=list)
    top_k: int = Field(default=4)

    class Config:
        arbitrary_types_allowed = True

    def _get_relevant_documents(self, query: str) -> List[Document]:
        """
        Tokenise the query the same way we tokenised the corpus,
        score every document with BM25, and return the top_k highest-scoring
        chunks wrapped in LangChain Document objects.
        """
        tokens = tokenize(query)
        scores = self.bm25.get_scores(tokens)
        top_indices = scores.argsort()[::-1][: self.top_k]

        results = []
        for idx in top_indices:
            doc = self.docs[idx]
            results.append(
                Document(
                    page_content=doc["text"],
                    metadata={"title": doc.get("title", ""),
                               "section": doc.get("section", ""),
                               "bm25_score": float(scores[idx])},
                )
            )
        return results


retriever = BM25Retriever(bm25=bm25_index, docs=bm25_corpus, top_k=4)


test_results = retriever.invoke("education girls Afghanistan")
print(f"Retriever smoke-test returned {len(test_results)} docs")
print(test_results[0].page_content[:200])

LangChain + Gemini RAG Chain

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain


llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.2,   # low temperature → more deterministic, fact-grounded answers
)


SYSTEM_PROMPT = """You are a precise question-answering assistant.
Answer the user's question using ONLY the information in the context below.
Do NOT use your own background knowledge — if the answer is not in the
context, say \"This information isn't in the given documnets.\"
Keep your answer concise (3–5 sentences).

Context:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
])


document_chain = create_stuff_documents_chain(llm, prompt)


rag_chain = create_retrieval_chain(retriever, document_chain)

print("RAG chain ready.")

In [ ]:
#functional test
test_q = "What restrictions have been placed on women's education in Afghanistan?"
test_ans = rag_chain.invoke({"input": test_q})

print("Question:", test_q)
print("\nAnswer:")
print(test_ans["answer"])
print("\nRetrieved from:")
for doc in test_ans["context"]:
    print(f"  [{doc.metadata.get('title', '')[:50]}] score={doc.metadata.get('bm25_score', 0):.2f}")

Latency & Accuracy Comparison

In [ ]:
import time

TEST_QUESTIONS = [
    "What restrictions have been placed on women's education in Afghanistan?",
    "How many girls are out of school in Afghanistan?",
    "What is UNICEF doing to support children in Afghanistan?",
    "How has the Taliban affected women's employment rights?",
    "What are the main human rights concerns in Afghanistan according to UN reports?",
]


full_context = "\n\n".join(art["text"] for art in corpus_raw)

naive_prompt = ChatPromptTemplate.from_messages([
    ("system", """Answer the question using ONLY the context below.
If the answer is not present say 'I cannot find that information.'

Context:\n{context}"""),
    ("human", "{input}"),
])

from langchain_core.runnables import RunnablePassthrough

naive_chain = (
    {"context": lambda _: full_context, "input": RunnablePassthrough()}
    | naive_prompt
    | llm
)

print(f"Full context length: {len(full_context):,} characters")
print(f"BM25 corpus size:    {len(bm25_corpus)} chunks")

In [ ]:

results = []

for q in TEST_QUESTIONS:
    print(f"\n{'='*60}")
    print(f"Q: {q}")

    t0 = time.time()
    rag_response = rag_chain.invoke({"input": q})
    rag_latency = time.time() - t0
    rag_answer = rag_response["answer"]

    t0 = time.time()
    naive_response = naive_chain.invoke(q)
    naive_latency = time.time() - t0
    naive_answer = naive_response.content

    results.append({
        "question": q,
        "rag_answer": rag_answer,
        "rag_latency": round(rag_latency, 2),
        "naive_answer": naive_answer,
        "naive_latency": round(naive_latency, 2),
    })

    print(f"  BM25-RAG  ({rag_latency:.1f}s): {rag_answer[:120]}...")
    print(f"  Naive     ({naive_latency:.1f}s): {naive_answer[:120]}...")

print("\nBenchmark complete.")

In [ ]:
print(f"{'Question':<55} {'BM25(s)':>8} {'Naive(s)':>9}")
print("-" * 74)

for r in results:
    q_short = r["question"][:52] + "..."
    print(f"{q_short:<55} {r['rag_latency']:>8.2f} {r['naive_latency']:>9.2f}")

avg_rag   = sum(r["rag_latency"]   for r in results) / len(results)
avg_naive = sum(r["naive_latency"] for r in results) / len(results)

print("-" * 74)
print(f"{'AVERAGE':<55} {avg_rag:>8.2f} {avg_naive:>9.2f}")
print(f"\nSpeedup: BM25-RAG is {avg_naive / avg_rag:.1f}x faster on average" if avg_rag > 0 else "")

In [ ]:
for i, r in enumerate(results, 1):
    print(f"\n{'#'*60}")
    print(f"QUESTION {i}: {r['question']}")
    print(f"\n[BM25-RAG — {r['rag_latency']}s]")
    print(r["rag_answer"])
    print(f"\n[Naive full-context — {r['naive_latency']}s]")
    print(r["naive_answer"])